# 02 — Splits, labels, and pre-registered decisions
**Goal:** freeze every evaluation decision *before any model exists*.

| # | Decision | Value | Rationale |
|---|----------|-------|-----------|
| 1 | Anomaly threshold θ | RUL < 30 | actionable maintenance window; ablated in nb 08 |
| 2 | Healthy reference | RUL > 120 | sensor-flat plateau (verified in nb 03) |
| 3 | Buffer band | 30 < RUL < 100 | ambiguous mid-life cycles excluded from binary eval |
| 4 | Protocols | buffer AND no-buffer | buffer saturates FD001 (z-s11 alone: 0.99) |
| 5 | Primary setting | FD004, regime-norm, no buffer | only setting with headroom |
| 6 | Seeds | {42,123,999,7,2026} | results as mean±std over engine-level splits |

**Split design:** engines, never rows (row splits leak unit identity). Three-way:
60% fit / 20% conformal calibration / 20% validation. Official `test_*` files are
touched once, in notebook 08.

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
from src.loaders import load, split_units, SEEDS, HEALTHY_RUL, THETA, BUFFER
from src import metrics

In [2]:
meta = {}
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    units = load(fd).unit.unique()
    meta[fd] = {str(seed): {k: v.tolist() for k, v in
                zip(['fit', 'cal', 'val'], split_units(units, seed))} for seed in SEEDS}
json.dump(meta, open('../data/processed/split_metadata.json', 'w'))
fit, cal, val = (meta['FD001']['42'][k] for k in ['fit', 'cal', 'val'])
print('FD001 seed42 -> fit/cal/val engines:', len(fit), len(cal), len(val))
print('disjoint:', set(fit).isdisjoint(cal) and set(fit).isdisjoint(val) and set(cal).isdisjoint(val))

FD001 seed42 -> fit/cal/val engines: 60 20 20
disjoint: True


## Label construction and class balance under both protocols

In [3]:
tr = load('FD001')
y_buf = metrics.labels(tr.rul, use_buffer=True)
y_nob = metrics.labels(tr.rul, use_buffer=False)
print('buffer protocol  : %5.1f%% anomalous, %5.1f%% normal, %5.1f%% excluded' %
      tuple(100 * np.mean(y_buf == v) for v in [1, 0, -1]))
print('no-buffer protocol: %5.1f%% anomalous, %4.1f%% normal' %
      tuple(100 * np.mean(y_nob == v) for v in [1, 0]))

buffer protocol  :  14.5% anomalous,  51.0% normal,  34.4% excluded
no-buffer protocol:  14.5% anomalous, 85.5% normal


~14.5% anomalous cycles (θ=30 over mean lifetime 206) — mild imbalance, PR curves stay readable.
The `src/metrics.py` module implements: AUROC, AUPRC (both label-based), cost-weighted error,
detection delay, and Spearman(score, RUL) as the label-free sanity check. Smoke test:

In [4]:
rng = np.random.RandomState(0)
fake_score = -tr.rul + rng.normal(0, 20, len(tr))   # noisy oracle
print('AUROC (buffer) of noisy oracle:', round(metrics.auroc(fake_score, tr.rul), 3))
print('Spearman(score, RUL):', round(metrics.spearman_rul(fake_score, tr.rul), 3))

AUROC (buffer) of noisy oracle: 1.0
Spearman(score, RUL): -0.959
